# Who Gets to Interrupt? Reputation-Calibrated Deference in Multi-Agent AI

**Research question.** When Evidence, Planning, and Idea Agents compete for limited human attention, how should an Attention Coordinator combine current importance and urgency claims with task-specific historical reliability to decide whether to interrupt, queue, or defer a message?

**Authorship and assistance.** Yiqiao Liu selected the research direction and central model idea. AI/Codex assisted with implementation and notebook construction.

## What this notebook demonstrates

This notebook uses the verified project implementation to compare current-claim and history-calibrated allocations of interruption authority. It connects current importance and urgency, task-specific historical reliability, routing decisions, attention cost, delayed warnings, and authority shares.

> **Academic and evidence boundary:** The 18 notifications are authored synthetic diagnostic cases. They are not participant data, real LLM outputs, field observations, or evidence of deployed safety performance. The heuristic $S=N\times R$ is proposed and inspectable, not proven optimal.

## 1. Locate the repository and import the reference implementation

When run inside the repository, this cell uses the existing checkout. In a fresh Google Colab runtime it clones the public repository from GitHub. During development it uses the current default branch; a final release can later display or pin an exact commit.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/micL1222/PS1-Yiqiao.git"

def find_repository(start: Path):
    candidates = [start, *start.parents, Path("/content/PS1-Yiqiao")]
    for candidate in candidates:
        if (candidate / "companion/src/scheduler.py").is_file():
            return candidate.resolve()
    return None

repository_root = find_repository(Path.cwd())
if repository_root is None:
    clone_root = Path("/content") if Path("/content").is_dir() else Path.cwd()
    clone_target = clone_root / "PS1-Yiqiao"
    if clone_target.exists():
        raise RuntimeError(f"{clone_target} exists but is not a valid PS1 checkout")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(clone_target)],
        check=True,
    )
    repository_root = find_repository(clone_target)

if repository_root is None:
    raise RuntimeError("Could not locate the PS1 repository")

os.chdir(repository_root)
companion_root = repository_root / "companion"
if str(companion_root) not in sys.path:
    sys.path.insert(0, str(companion_root))

try:
    commit = subprocess.run(
        ["git", "-C", str(repository_root), "rev-parse", "HEAD"],
        check=True, capture_output=True, text=True
    ).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    commit = "unavailable"

print(f"Repository ready: {repository_root.name}")
print(f"Commit used: {commit}")

Repository ready: PS1-Yiqiao
Commit used: 343fb3e4ba3bcdc3d030831a32ab16f75a51d84b


In [2]:
from collections import Counter
from IPython.display import Markdown, display

from src.scheduler import (
    COLD_START_RELIABILITY,
    CURRENT_CLAIM_POLICY,
    REPUTATION_POLICY,
    build_reliability_lookup,
    equal_count_ranking,
    evaluate_decisions,
    load_json,
    reliability_for,
    schedule_reports,
    swap_agent_reliabilities,
)

def show_table(headers, rows):
    text_rows = [[str(value) for value in row] for row in rows]
    header = "| " + " | ".join(headers) + " |"
    divider = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in text_rows]
    display(Markdown("\n".join([header, divider, *body])))

## 2. Model definitions

For importance $I$ and urgency $U$, with $\alpha=0.5$:

$$N=0.5I+0.5U$$

- **Current-claim policy:** $S=N$
- **Reputation-calibrated policy:** $S=N\times R$
- **Beta(1,1) reliability:** $R=(\text{valid}+1)/(\text{audited}+2)$

Historical reliability is task-specific. An unseen agent-task pair receives the transparent cold-start value $R=0.50$. Multiplication is a simple proposed heuristic, not a proof of optimal allocation.

## 3. Online routing policy

- $S\geq0.65$: interrupt immediately.
- $0.35\leq S<0.65$: queue.
- $S<0.35$: deliver in the minute-20 digest.

A queued report is delivered at the **strictly next** five-minute checkpoint. Thus minute 4 maps to 5, minute 5 maps to 10, minute 9 maps to 10, and minute 10 maps to 15. Reports arrive before the minute-20 session horizon.

## 4. Load and inspect the authored synthetic reports

This cell loads only the fields available to the online coordinator. Hidden validity and loss labels are loaded later, after decisions have been made.

In [3]:
reports_document = load_json(companion_root / "data/reports.json")
history_document = load_json(companion_root / "data/history.json")
reports = reports_document["reports"]

assert reports_document["metadata"]["synthetic"] is True
assert len(reports) == 18
assert not any(
    hidden in report
    for report in reports
    for hidden in ("valid", "late_loss", "latest_useful_minute")
)

show_table(
    ["ID", "Agent", "Task type", "Arrival", "Importance", "Urgency"],
    [
        (
            report["id"], report["agent"], report["task_type"],
            report["arrival_minute"], report["importance"], report["urgency"]
        )
        for report in reports
    ],
)

| ID | Agent | Task type | Arrival | Importance | Urgency |
| --- | --- | --- | --- | --- | --- |
| E01 | Evidence | evidence_check | 1 | 0.9 | 0.9 |
| E02 | Evidence | evidence_check | 4 | 0.7 | 0.6 |
| E03 | Evidence | evidence_check | 5 | 0.65 | 0.648 |
| E04 | Evidence | evidence_check | 9 | 0.4 | 0.3 |
| E05 | Evidence | evidence_check | 11 | 0.35 | 0.348 |
| E06 | Evidence | evidence_check | 14 | 0.8 | 0.7 |
| P01 | Planning | planning_risk | 2 | 0.95 | 0.85 |
| P02 | Planning | planning_risk | 5 | 0.7 | 0.6 |
| P03 | Planning | planning_risk | 6 | 0.5 | 0.4 |
| P04 | Planning | planning_risk | 10 | 0.35 | 0.35 |
| P05 | Planning | planning_risk | 13 | 0.3 | 0.398 |
| P06 | Planning | planning_risk | 17 | 0.8 | 0.6 |
| I01 | Idea | idea_suggestion | 3 | 1.0 | 0.9 |
| I02 | Idea | idea_suggestion | 4 | 0.8 | 0.8 |
| I03 | Idea | idea_suggestion | 5 | 0.7 | 0.6 |
| I04 | Idea | idea_suggestion | 9 | 0.4 | 0.3 |
| I05 | Idea | idea_suggestion | 10 | 0.35 | 0.348 |
| I06 | Idea | idea_suggestion | 18 | 0.9 | 0.7 |

## 5. Verify task-specific historical reliability

The displayed values are computed from the audited-count inputs with Beta(1,1) smoothing. They are synthetic model inputs, not estimates from deployed agents.

In [4]:
reliabilities = build_reliability_lookup(history_document)
agent_task = {
    "Evidence": "evidence_check",
    "Planning": "planning_risk",
    "Idea": "idea_suggestion",
}
reliability_rows = [
    (agent, task_type, f"{reliabilities[(agent, task_type)]:.2f}")
    for agent, task_type in agent_task.items()
]
cold_start = reliability_for(
    {"agent": "Evidence", "task_type": "unseen_task"}, reliabilities
)
reliability_rows.append(("Cold start", "unseen agent-task pair", f"{cold_start:.2f}"))
show_table(["Agent", "Task type", "R"], reliability_rows)

| Agent | Task type | R |
| --- | --- | --- |
| Evidence | evidence_check | 0.90 |
| Planning | planning_risk | 0.50 |
| Idea | idea_suggestion | 0.20 |
| Cold start | unseen agent-task pair | 0.50 |

## 6. Run the current-claim online policy

In [5]:
current_decisions = schedule_reports(reports, CURRENT_CLAIM_POLICY, reliabilities)
show_table(
    ["ID", "Agent", "N", "S", "Route", "Delivery"],
    [
        (d.id, d.agent, f"{d.current_claim_score:.3f}", f"{d.policy_score:.3f}", d.route, d.delivery_minute)
        for d in current_decisions
    ],
)

| ID | Agent | N | S | Route | Delivery |
| --- | --- | --- | --- | --- | --- |
| E01 | Evidence | 0.900 | 0.900 | interrupt | 1 |
| E02 | Evidence | 0.650 | 0.650 | interrupt | 4 |
| E03 | Evidence | 0.649 | 0.649 | queue | 10 |
| E04 | Evidence | 0.350 | 0.350 | queue | 10 |
| E05 | Evidence | 0.349 | 0.349 | digest | 20 |
| E06 | Evidence | 0.750 | 0.750 | interrupt | 14 |
| P01 | Planning | 0.900 | 0.900 | interrupt | 2 |
| P02 | Planning | 0.650 | 0.650 | interrupt | 5 |
| P03 | Planning | 0.450 | 0.450 | queue | 10 |
| P04 | Planning | 0.350 | 0.350 | queue | 15 |
| P05 | Planning | 0.349 | 0.349 | digest | 20 |
| P06 | Planning | 0.700 | 0.700 | interrupt | 17 |
| I01 | Idea | 0.950 | 0.950 | interrupt | 3 |
| I02 | Idea | 0.800 | 0.800 | interrupt | 4 |
| I03 | Idea | 0.650 | 0.650 | interrupt | 5 |
| I04 | Idea | 0.350 | 0.350 | queue | 10 |
| I05 | Idea | 0.349 | 0.349 | digest | 20 |
| I06 | Idea | 0.800 | 0.800 | interrupt | 18 |

## 7. Run the reputation-calibrated online policy

In [6]:
reputation_decisions = schedule_reports(reports, REPUTATION_POLICY, reliabilities)
show_table(
    ["ID", "Agent", "N", "R", "S=N×R", "Route", "Delivery"],
    [
        (
            d.id, d.agent, f"{d.current_claim_score:.3f}",
            f"{d.historical_reliability:.2f}", f"{d.policy_score:.3f}",
            d.route, d.delivery_minute
        )
        for d in reputation_decisions
    ],
)

| ID | Agent | N | R | S=N×R | Route | Delivery |
| --- | --- | --- | --- | --- | --- | --- |
| E01 | Evidence | 0.900 | 0.90 | 0.810 | interrupt | 1 |
| E02 | Evidence | 0.650 | 0.90 | 0.585 | queue | 5 |
| E03 | Evidence | 0.649 | 0.90 | 0.584 | queue | 10 |
| E04 | Evidence | 0.350 | 0.90 | 0.315 | digest | 20 |
| E05 | Evidence | 0.349 | 0.90 | 0.314 | digest | 20 |
| E06 | Evidence | 0.750 | 0.90 | 0.675 | interrupt | 14 |
| P01 | Planning | 0.900 | 0.50 | 0.450 | queue | 5 |
| P02 | Planning | 0.650 | 0.50 | 0.325 | digest | 20 |
| P03 | Planning | 0.450 | 0.50 | 0.225 | digest | 20 |
| P04 | Planning | 0.350 | 0.50 | 0.175 | digest | 20 |
| P05 | Planning | 0.349 | 0.50 | 0.174 | digest | 20 |
| P06 | Planning | 0.700 | 0.50 | 0.350 | queue | 20 |
| I01 | Idea | 0.950 | 0.20 | 0.190 | digest | 20 |
| I02 | Idea | 0.800 | 0.20 | 0.160 | digest | 20 |
| I03 | Idea | 0.650 | 0.20 | 0.130 | digest | 20 |
| I04 | Idea | 0.350 | 0.20 | 0.070 | digest | 20 |
| I05 | Idea | 0.349 | 0.20 | 0.070 | digest | 20 |
| I06 | Idea | 0.800 | 0.20 | 0.160 | digest | 20 |

## 8. Evaluate the two online policies

Only now do we load the separate, evaluation-only outcomes. An **important alert** is defined as valid with `late_loss >= 7`. It is timely when delivered no later than its authored `latest_useful_minute`. Attention costs are 2.0 for interrupt, 0.2 for queue, and 0.05 for digest; joint loss is attention cost plus late-alert loss.

In [7]:
outcomes_document = load_json(companion_root / "data/outcomes.json")
outcomes = outcomes_document["outcomes"]
assert outcomes_document["metadata"]["evaluation_only"] is True

current_metrics = evaluate_decisions(current_decisions, outcomes)
reputation_metrics = evaluate_decisions(reputation_decisions, outcomes)

metric_specs = [
    ("Immediate interruptions", "immediate_interruptions"),
    ("Queued notices", "queued_notices"),
    ("Digest notices", "digest_notices"),
    ("Invalid immediate notices", "invalid_immediate_notices"),
    ("Invalid interruption fraction", "invalid_interruption_fraction"),
    ("Important alerts", "important_alert_count"),
    ("Important alerts on time", "important_alerts_delivered_on_time"),
    ("Important-alert timely recall", "important_alert_timely_recall"),
    ("Attention cost", "attention_cost"),
    ("Late-alert loss", "late_alert_loss"),
    ("Synthetic joint loss", "synthetic_joint_loss"),
]
show_table(
    ["Metric", "Current claim", "Reputation calibrated"],
    [(label, current_metrics[key], reputation_metrics[key]) for label, key in metric_specs],
)

| Metric | Current claim | Reputation calibrated |
| --- | --- | --- |
| Immediate interruptions | 10 | 2 |
| Queued notices | 5 | 4 |
| Digest notices | 3 | 12 |
| Invalid immediate notices | 4 | 1 |
| Invalid interruption fraction | 0.4 | 0.5 |
| Important alerts | 8 | 8 |
| Important alerts on time | 8 | 2 |
| Important-alert timely recall | 1.0 | 0.25 |
| Attention cost | 21.15 | 5.4 |
| Late-alert loss | 0.0 | 49.0 |
| Synthetic joint loss | 21.15 | 54.4 |

In [8]:
# Regression checks against the verified deterministic computational core.
assert (
    current_metrics["immediate_interruptions"],
    current_metrics["queued_notices"],
    current_metrics["digest_notices"],
) == (10, 5, 3)
assert current_metrics["invalid_immediate_notices"] == 4
assert current_metrics["invalid_interruption_fraction"] == 0.40
assert current_metrics["important_alerts_delivered_on_time"] == 8
assert current_metrics["important_alert_timely_recall"] == 1.00
assert current_metrics["attention_cost"] == 21.15
assert current_metrics["late_alert_loss"] == 0.00
assert current_metrics["synthetic_joint_loss"] == 21.15

assert (
    reputation_metrics["immediate_interruptions"],
    reputation_metrics["queued_notices"],
    reputation_metrics["digest_notices"],
) == (2, 4, 12)
assert reputation_metrics["invalid_immediate_notices"] == 1
assert reputation_metrics["invalid_interruption_fraction"] == 0.50
assert reputation_metrics["important_alerts_delivered_on_time"] == 2
assert reputation_metrics["important_alert_timely_recall"] == 0.25
assert reputation_metrics["attention_cost"] == 5.40
assert reputation_metrics["late_alert_loss"] == 49.00
assert reputation_metrics["synthetic_joint_loss"] == 54.40

## 9. Interpret the synthetic tradeoff

The next cell states only comparisons computed above. The relevant distinction is between the **absolute number** of invalid immediate notices and their **fraction among all immediate interruptions**.

In [9]:
print(
    "In these authored synthetic cases, the multiplicative reputation policy "
    f"reduces attention cost from {current_metrics['attention_cost']:.2f} to "
    f"{reputation_metrics['attention_cost']:.2f}, but timely important-alert recall "
    f"falls from {current_metrics['important_alert_timely_recall']:.2f} to "
    f"{reputation_metrics['important_alert_timely_recall']:.2f} and joint loss rises "
    f"from {current_metrics['synthetic_joint_loss']:.2f} to "
    f"{reputation_metrics['synthetic_joint_loss']:.2f}."
)
print(
    f"Invalid immediate notices fall in absolute number from "
    f"{current_metrics['invalid_immediate_notices']} to "
    f"{reputation_metrics['invalid_immediate_notices']}, while their fraction among "
    f"immediate interruptions rises from "
    f"{current_metrics['invalid_interruption_fraction']:.2f} to "
    f"{reputation_metrics['invalid_interruption_fraction']:.2f}."
)

In these authored synthetic cases, the multiplicative reputation policy reduces attention cost from 21.15 to 5.40, but timely important-alert recall falls from 1.00 to 0.25 and joint loss rises from 21.15 to 54.40.
Invalid immediate notices fall in absolute number from 4 to 1, while their fraction among immediate interruptions rises from 0.40 to 0.50.


## 10. History-swap stress test

This imposed **offline synthetic stress test** swaps the Evidence and Idea reliability levels while preserving every current claim and hidden outcome. It is a sensitivity diagnostic, not an estimate of real distribution shift.

In [10]:
swapped_reliabilities = swap_agent_reliabilities(reliabilities, "Evidence", "Idea")
swapped_current_decisions = schedule_reports(
    reports, CURRENT_CLAIM_POLICY, swapped_reliabilities
)
swapped_reputation_decisions = schedule_reports(
    reports, REPUTATION_POLICY, swapped_reliabilities
)
swapped_current_metrics = evaluate_decisions(swapped_current_decisions, outcomes)
swapped_reputation_metrics = evaluate_decisions(swapped_reputation_decisions, outcomes)

def operational_signature(decisions):
    return [(d.id, d.policy_score, d.route, d.delivery_minute) for d in decisions]

assert operational_signature(current_decisions) == operational_signature(swapped_current_decisions)
show_table(
    ["Metric", "Current claim after swap", "Reputation after swap"],
    [
        ("Immediate interruptions", swapped_current_metrics["immediate_interruptions"], swapped_reputation_metrics["immediate_interruptions"]),
        ("Queued notices", swapped_current_metrics["queued_notices"], swapped_reputation_metrics["queued_notices"]),
        ("Digest notices", swapped_current_metrics["digest_notices"], swapped_reputation_metrics["digest_notices"]),
        ("Important-alert timely recall", swapped_current_metrics["important_alert_timely_recall"], swapped_reputation_metrics["important_alert_timely_recall"]),
        ("Synthetic joint loss", swapped_current_metrics["synthetic_joint_loss"], swapped_reputation_metrics["synthetic_joint_loss"]),
    ],
)

assert (
    swapped_reputation_metrics["immediate_interruptions"],
    swapped_reputation_metrics["queued_notices"],
    swapped_reputation_metrics["digest_notices"],
) == (3, 3, 12)
assert swapped_reputation_metrics["important_alert_timely_recall"] == 0.125
assert swapped_reputation_metrics["synthetic_joint_loss"] == 66.20

| Metric | Current claim after swap | Reputation after swap |
| --- | --- | --- |
| Immediate interruptions | 10 | 3 |
| Queued notices | 5 | 3 |
| Digest notices | 3 | 12 |
| Important-alert timely recall | 1.0 | 0.125 |
| Synthetic joint loss | 21.15 | 66.2 |

## 11. Corrected equal-count offline ranking diagnostic

This is **not an online deployment policy**. Both policies receive the same ten immediate-interruption slots. Each policy ranks reports by its own score; the top ten interrupt, and every non-selected report uses the same fallback route: queue to the strictly next five-minute checkpoint. This isolates the allocation of interruption authority from mechanical score shrinkage and queue-versus-digest differences.

In [11]:
immediate_budget = current_metrics["immediate_interruptions"]
equal_current = equal_count_ranking(
    reports, CURRENT_CLAIM_POLICY, reliabilities, immediate_budget
)
equal_reputation = equal_count_ranking(
    reports, REPUTATION_POLICY, reliabilities, immediate_budget
)
equal_current_metrics = evaluate_decisions(equal_current, outcomes)
equal_reputation_metrics = evaluate_decisions(equal_reputation, outcomes)
current_selected = [d.id for d in equal_current if d.route == "interrupt"]
reputation_selected = [d.id for d in equal_reputation if d.route == "interrupt"]

show_table(
    ["Metric", "Current-claim ranking", "Reputation ranking"],
    [
        ("Immediate interruptions", equal_current_metrics["immediate_interruptions"], equal_reputation_metrics["immediate_interruptions"]),
        ("Queued notices", equal_current_metrics["queued_notices"], equal_reputation_metrics["queued_notices"]),
        ("Digest notices", equal_current_metrics["digest_notices"], equal_reputation_metrics["digest_notices"]),
        ("Attention cost", equal_current_metrics["attention_cost"], equal_reputation_metrics["attention_cost"]),
        ("Important-alert timely recall", equal_current_metrics["important_alert_timely_recall"], equal_reputation_metrics["important_alert_timely_recall"]),
        ("Synthetic joint loss", equal_current_metrics["synthetic_joint_loss"], equal_reputation_metrics["synthetic_joint_loss"]),
    ],
)
print("Current-claim selected IDs:", ", ".join(current_selected))
print("Reputation selected IDs:", ", ".join(reputation_selected))

| Metric | Current-claim ranking | Reputation ranking |
| --- | --- | --- |
| Immediate interruptions | 10 | 10 |
| Queued notices | 8 | 8 |
| Digest notices | 0 | 0 |
| Attention cost | 21.6 | 21.6 |
| Important-alert timely recall | 1.0 | 1.0 |
| Synthetic joint loss | 21.6 | 21.6 |

Current-claim selected IDs: E01, E02, E06, P01, P02, P06, I01, I02, I03, I06
Reputation selected IDs: E01, E02, E03, E04, E05, E06, P01, P02, P03, P06


In [12]:
expected_current_selected = [
    "E01", "E02", "E06", "P01", "P02",
    "P06", "I01", "I02", "I03", "I06",
]
expected_reputation_selected = [
    "E01", "E02", "E03", "E04", "E05",
    "E06", "P01", "P02", "P03", "P06",
]
assert current_selected == expected_current_selected
assert reputation_selected == expected_reputation_selected
for metrics in (equal_current_metrics, equal_reputation_metrics):
    assert (
        metrics["immediate_interruptions"],
        metrics["queued_notices"],
        metrics["digest_notices"],
    ) == (10, 8, 0)
    assert metrics["attention_cost"] == 21.60
    assert metrics["important_alert_timely_recall"] == 1.00
    assert metrics["synthetic_joint_loss"] == 21.60

authority_rows = []
for agent in ("Evidence", "Planning", "Idea"):
    authority_rows.append((
        agent,
        equal_current_metrics["routing_by_agent"][agent]["interrupt_count"],
        equal_reputation_metrics["routing_by_agent"][agent]["interrupt_count"],
    ))
show_table(
    ["Agent", "Current-claim immediate slots", "Reputation immediate slots"],
    authority_rows,
)

| Agent | Current-claim immediate slots | Reputation immediate slots |
| --- | --- | --- |
| Evidence | 3 | 6 |
| Planning | 3 | 4 |
| Idea | 4 | 0 |

At equal interruption count and with identical fallback handling, the main visible difference is the allocation of interruption authority. The current-claim ranking distributes selected slots across Evidence, Planning, and Idea. The reputation ranking concentrates them on Evidence and Planning and gives Idea zero immediate slots. The two policies have equal joint loss in this diagnostic; reputation does not improve it.

## 12. Limitations

- The reports and histories are constructed diagnostic cases.
- Thresholds, attention costs, loss values, and useful-by times are assumed.
- There are no human participants and no real AI agents.
- The model provides no truthful-reporting or strategic-equilibrium proof.
- There is no deployment or safety validation.
- Historical deference can create path dependence or self-reinforcing silence: fewer interruptions can mean fewer opportunities to demonstrate validity.
- Task-specific reliability can become stale under domain or task shifts.
- These results diagnose this fixed scenario; they do not establish that either policy generalizes or is optimal.

## 13. Final reproducibility checks

The final cell repeats the essential executable checks. It prints success only if every assertion passes.

In [13]:
assert len(reports) == 18
assert reliabilities[("Evidence", "evidence_check")] == 0.90
assert reliabilities[("Planning", "planning_risk")] == 0.50
assert reliabilities[("Idea", "idea_suggestion")] == 0.20
assert cold_start == COLD_START_RELIABILITY == 0.50
assert (current_metrics["immediate_interruptions"], current_metrics["queued_notices"], current_metrics["digest_notices"]) == (10, 5, 3)
assert (reputation_metrics["immediate_interruptions"], reputation_metrics["queued_notices"], reputation_metrics["digest_notices"]) == (2, 4, 12)
assert operational_signature(current_decisions) == operational_signature(swapped_current_decisions)
assert (swapped_reputation_metrics["immediate_interruptions"], swapped_reputation_metrics["queued_notices"], swapped_reputation_metrics["digest_notices"]) == (3, 3, 12)
assert swapped_reputation_metrics["important_alert_timely_recall"] == 0.125
assert swapped_reputation_metrics["synthetic_joint_loss"] == 66.20
assert (equal_current_metrics["immediate_interruptions"], equal_current_metrics["queued_notices"], equal_current_metrics["digest_notices"]) == (10, 8, 0)
assert (equal_reputation_metrics["immediate_interruptions"], equal_reputation_metrics["queued_notices"], equal_reputation_metrics["digest_notices"]) == (10, 8, 0)
assert current_selected == expected_current_selected
assert reputation_selected == expected_reputation_selected
print("All notebook checks passed.")

All notebook checks passed.
